In [127]:
%pip install nltk
%pip install websockets
%pip install playwright
%pip install transformers torch
%pip install spacy
%pip install asyncpg



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [128]:
import os
import json
import asyncio
import websockets
import nltk
import spacy
import asyncpg
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from playwright.async_api import async_playwright
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline
from datetime import datetime

In [ ]:
# Resource setup
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Download VADER lexicon
nltk.download('vader_lexicon')


sia = SentimentIntensityAnalyzer()

nltk.download(['punkt', 'stopwords', 'punkt_tab', 'wordnet', 'omw-1.4'])
lemmatizer = WordNetLemmatizer()

nlp = spacy.load("en_core_web_sm")


sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device= -1 # Set to 0 if you have a GPU
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [130]:
#BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
AUTH_FILE = "auth.json"
MAX_CONCURRENT_TABS = 1
semaphore = asyncio.Semaphore(MAX_CONCURRENT_TABS)
word_counts = Counter()

# State management
is_scraping = asyncio.Lock() # Ensures only one browser session at a time
background_subreddits = ["Mumbai", "India", "AskIndianWomen", "LegalAdviceIndia", "Munich", "BoycottIsrael"]
SCRAPE_INTERVAL = 900 # 15 minutes

# Refined ignore list
ignore_words = [
    "i", "me", "my", "myself", "we", "us", "our", "ours", "ourselves", "you", "your", "yours", 
    "yourself", "yourselves", "he", "him", "his", "she", "her", "hers", "it", "its", "they", 
    "them", "their", "someone", "anyone", "am", "is", "are", "was", "were", "be", "been", 
    "being", "have", "has", "had", "do", "does", "did", "shall", "will", "should", "would", 
    "may", "might", "must", "can", "could", "get", "got", "getting", "go", "goes", "went", 
    "gone", "take", "took", "taken", "make", "made", "making", "like", "know", "knows", 
    "knew", "want", "wants", "wanted", "look", "looks", "really", "very", "just", "actually", 
    "basically", "literally", "even", "also", "still", "maybe", "probably", "quite", "much", 
    "many", "lot", "lots", "thing", "things", "stuff", "reddit", "subreddit", "sub", "post", 
    "posts", "comment", "comments", "thread", "edit", "deleted", "removed", "amp", "https", 
    "http", "com", "www"
]

In [131]:
async def safe_send(ws, data):
    """Safely sends data only if a UI user is actively connected."""
    if ws and ws.state.name == 'OPEN':
        try:
            await ws.send(json.dumps(data))
        except Exception as e:
            print(f"WebSocket send error: {e}")

In [132]:
async def reanalyze_cache(subreddit):
    """
    Fetches all posts for a subreddit from PostgreSQL, 
    re-runs NLP analysis, and updates the records.
    """
    print(f"Starting database re-analysis for r/{subreddit}...")
    
    conn = await get_db_connection()
    try:
        # 1. Fetch all posts for this sub (No limit, we want to upgrade everything)
        rows = await conn.fetch(
            "SELECT id, title, body FROM reddit_posts WHERE subreddit = $1", 
            subreddit
        )
        total = len(rows)
        processed = 0
        
        for row in rows:
            # Combine title and body for full context
            title = row.get('title', '')
            body = row.get('body', '')
            full_text = f"{title} {body}"
            
            # 2. Re-run NLP Analysis using latest models
            sentiment = get_sentiment(full_text)
            
            # Keywords Extraction
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [
                lemmatizer.lemmatize(w) for w in words 
                if w.isalnum() and w not in sw and w not in ignore_words
            ]
            keywords = dict(Counter(clean))
            
            # NER Extraction (All 18 labels)
            entities = extract_entities(full_text)
            
            # 3. Update the record in the Database
            # Using json.dumps ensures the driver passes the correct type to JSONB columns
            await conn.execute('''
                UPDATE reddit_posts 
                SET sentiment = $1, keywords = $2, entities = $3 
                WHERE id = $4
            ''', sentiment, json.dumps(keywords), json.dumps(entities), row['id'])
            
            processed += 1
            if processed % 10 == 0:
                print(f"Progress: {processed}/{total} posts updated in DB...")
                
    finally:
        await conn.close()
        
    print(f"Re-analysis complete for r/{subreddit}. All data is now standardized.")

# Example Usage (Manually run this for any sub you already have data for):
# await reanalyze_cache("AskIndianWomen")

In [133]:
def extract_entities(text):
    doc = nlp(text)
    entities = []
    # We filter for categories most useful for Business Intelligence
    target_labels = [
        "PERSON", "NORP", "FAC", "ORG", "GPE", "LOC", "PRODUCT", "EVENT", 
        "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT", 
        "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"
    ]
    
    for ent in doc.ents:
        if ent.label_ in target_labels:
            entities.append({
                "text": ent.text,
                "label": ent.label_
            })
    return entities

In [134]:
def get_sentiment(text):
    """
    Scans entire text and understands context using a 
    Transformer model. Returns Positive, Neutral, or Negative.
    """
    # Truncate text to 512 tokens (Model limit)
    truncated_text = text[:512]
    result = sentiment_pipeline(truncated_text)[0]
    
    # Mapping the model labels to your format
    label_map = {
        "positive": "Positive",
        "neutral": "Neutral",
        "negative": "Negative"
    }
    return label_map.get(result['label'].lower(), "Neutral")

In [135]:


DB_CONFIG = {
    "user": "postgres",
    "password": "admin123",
    "database": "reddit-scrapper",
    "host": "127.0.0.1"
}

def safe_parse_timestamp(ts_str):
    """
    Converts Reddit ISO strings to Python datetime objects.
    Strips timezone info to prevent 'offset-naive' vs 'offset-aware' errors.
    """
    if not ts_str or ts_str == "" or ts_str == "None":
        return None
    try:
        # 1. Parse the ISO string (making it 'aware' initially)
        dt = datetime.fromisoformat(ts_str.replace('Z', '+00:00'))
        
        # 2. Strip the timezone info to make it 'naive' 
        # This matches PostgreSQL's TIMESTAMP WITHOUT TIME ZONE
        return dt.replace(tzinfo=None)
    except Exception as e:
        print(f"Timestamp Parse Error: {e}")
        return None

async def get_db_connection():
    return await asyncpg.connect(**DB_CONFIG)

async def load_posts_from_db(subreddit, limit):
    conn = await get_db_connection()
    try:
        rows = await conn.fetch(
            "SELECT * FROM reddit_posts WHERE subreddit = $1 ORDER BY timestamp DESC LIMIT $2",
            subreddit, limit
        )
        
        data_dict = {}
        for row in rows:
            d = dict(row)
            # 1. Convert timestamp back to string for frontend
            if d.get('timestamp'):
                d['timestamp'] = d['timestamp'].isoformat()
            
            # 2. CRITICAL FIX: Ensure keywords and entities are objects/arrays, not strings
            if isinstance(d.get('keywords'), str):
                d['keywords'] = json.loads(d['keywords'])
            if isinstance(d.get('entities'), str):
                d['entities'] = json.loads(d['entities'])
                
            data_dict[d['id']] = d
        return data_dict
    finally:
        await conn.close()

async def save_post_to_db(post_entry, subreddit):
    conn = await get_db_connection()
    try:
        ts_obj = safe_parse_timestamp(post_entry.get('timestamp'))
        
        await conn.execute('''
            INSERT INTO reddit_posts (id, subreddit, timestamp, title, body, sentiment, keywords, entities)
            VALUES ($1, $2, $3, $4, $5, $6, $7, $8)
            ON CONFLICT (id) DO UPDATE SET
                sentiment = EXCLUDED.sentiment,
                keywords = EXCLUDED.keywords, 
                entities = EXCLUDED.entities
        ''', 
        post_entry['id'], 
        subreddit, 
        ts_obj, 
        post_entry['title'], 
        post_entry['body'], 
        post_entry['sentiment'],
        # Use json.dumps here to satisfy the driver; 
        # Postgres will convert it to JSONB automatically
        json.dumps(post_entry['keywords']), 
        json.dumps(post_entry['entities'])) 
    finally:
        await conn.close()

In [136]:
async def migrate_data():
    cache_dir = "cache"
    if not os.path.exists(cache_dir): return
    
    conn = await get_db_connection()
    print("Migrating JSON files to PostgreSQL...")
    
    for filename in os.listdir(cache_dir):
        if filename.endswith(".json"):
            sub = filename.replace(".json", "")
            with open(os.path.join(cache_dir, filename), "r", encoding="utf-8") as f:
                data = json.load(f)
                for pid, post in data.items():
                    ts_obj = safe_parse_timestamp(post.get('timestamp'))
                    try:
                        await conn.execute('''
                            INSERT INTO reddit_posts (id, subreddit, timestamp, title, body, sentiment, keywords, entities, last_cached)
                            VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9)
                            ON CONFLICT (id) DO NOTHING
                        ''', 
                        pid, sub, ts_obj, post.get('title'), 
                        post.get('body'), post.get('sentiment'), 
                        json.dumps(post.get('keywords', {})), # String for the driver
                        json.dumps(post.get('entities', [])), # String for the driver
                        post.get('last_cached', False))
                    except Exception as e:
                        print(f"Error migrating post {pid}: {e}")
            print(f"Finished r/{sub}")
            
    await conn.close()
    print("Migration Success!")


#await migrate_data()

In [137]:
async def get_post_metadata(post):
    pid = await post.get_attribute("id")
    time_loc = post.locator("time").first
    ts = ""
    if await time_loc.count() > 0:
        ts = await time_loc.get_attribute("datetime")
    return pid, ts

async def scrape_single_post(context, url, pid, ts, websocket, total_priority, tracker, cache, subreddit, is_priority):
    async with semaphore:
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            
            title = "[Title Not Found]"
            for sel in ['h1[slot="title"]', 'h1[id^="post-title-"]', 'shreddit-title', 'h1']:
                loc = page.locator(sel).first
                if await loc.is_visible(timeout=3000):
                    title = await loc.inner_text(); break

            content = ""
            for sel in ['shreddit-post-text-body', 'div[slot="text-body"]']:
                loc = page.locator(sel).first
                if await loc.count() > 0:
                    content = await loc.inner_text(timeout=3000)
                    if content: break
            
            # NLP & Sentiment
            full_text = title + " " + content
            sentiment = get_sentiment(full_text)
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [lemmatizer.lemmatize(w) for w in words if w.isalnum() and w not in sw and w not in ignore_words]
            keywords = dict(Counter(clean))
            
            # --- PERSIST TO CACHE IMMEDIATELY ---
            post_entry = {
                "id": pid, 
                "timestamp": ts, 
                "title": title, 
                "body": content, 
                "keywords": keywords, 
                "sentiment": sentiment,
                "entities": extract_entities(full_text)
            }
            cache[pid] = post_entry
            await save_post_to_db(post_entry, subreddit)

            # --- SEND DELTAS ONLY FOR PRIORITY NEW POSTS ---
            if is_priority:
                tracker['ui_processed'] += 1
                progress = int((tracker['ui_processed'] / total_priority) * 100) if total_priority > 0 else 100
                
                if websocket.state.name == 'OPEN':
                    # Send the full post object directly
                    await websocket.send(json.dumps({
                        "type": "delta_update",
                        "post": post_entry,
                        "progress": progress
                    }))
                    print(f"New Post Scraped: {title[:30]} | Progress: {progress}%")
            else:
                print(f"Background Cached (Gap Fill): {title[:30]}")
            
            return post_entry
        except Exception as e:
            print(f"Error {url}: {e}"); return None
        finally: await page.close()

In [138]:
async def run_scraper(websocket, subreddit, count, use_only_cache=False):
    count = int(count)
    cache = await load_posts_from_db(subreddit, count)
    
    # --- CACHE ONLY MODE ---
    if use_only_cache:
        print(f"Cache-only mode requested for r/{subreddit}.")
        await safe_send(websocket, {"type": "status", "message": f"Loading cached data for r/{subreddit}..."})
        final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
        await safe_send(websocket, {"type": "progress", "value": 100})
        await safe_send(websocket, {"type": "final_data", "posts": final_selection})
        return 

    # --- ACTIVE SCRAPE MODE ---
    is_ui_request = websocket is not None
    if is_ui_request:
        await safe_send(websocket, {"type": "status", "message": f"Fetching latest posts from r/{subreddit}..."})

    sorted_cache_list = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)
    safe_last_id = next((pid for pid, d in cache.items() if d.get("last_cached")), None)
    if not safe_last_id and sorted_cache_list: 
        safe_last_id = sorted_cache_list[0]['id']
    
    playwright = await async_playwright().start()
    # If no websocket (background task), run completely headless to save RAM
    browser = await playwright.chromium.launch(headless=False)
    context = await browser.new_context(**({"storage_state": AUTH_FILE} if os.path.exists(AUTH_FILE) else {}))
    
    try:
        if is_ui_request:
            await safe_send(websocket, {"type": "status", "message": f"Opening r/{subreddit} in background browser..."})
            
        main_page = await context.new_page()
        await main_page.goto(f"https://www.reddit.com/r/{subreddit}/new", wait_until="domcontentloaded")
        
        launched_ids = set()
        tracker = {'ui_processed': 0}
        top_post_id = None
        scrape_tasks = []
        found_checkpoint = False
        
        # Scroll and collect links until we hit our limit OR find a post we already have
        while not found_checkpoint and len(scrape_tasks) < count:
            posts = await main_page.locator('shreddit-post').all()
            new_on_page = False
            
            for p in posts:
                if len(scrape_tasks) >= count:
                    break
                    
                pid, ts = await get_post_metadata(p)
                if not pid or pid in launched_ids: 
                    continue
                    
                if not top_post_id: 
                    top_post_id = pid
                
                launched_ids.add(pid)
                new_on_page = True

                # Stop if we hit a post we already scraped previously
                if pid == safe_last_id:
                    print("Found previously cached post. Stopping scroll.")
                    found_checkpoint = True
                    break

                # If it's a new post, queue it up for deep scraping
                if pid not in cache:
                    href = await p.locator('a[slot="full-post-link"]').first.get_attribute('href')
                    if href:
                        url = f"https://www.reddit.com{href}"
                        scrape_tasks.append(
                            scrape_single_post(context, url, pid, ts, websocket, count, tracker, cache, subreddit, is_priority=is_ui_request)
                        )

            if not found_checkpoint and len(scrape_tasks) < count:
                if new_on_page:
                    await main_page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await main_page.wait_for_timeout(2000)
                else:
                    break # No more posts loading

        # Execute all gathered scrape tasks concurrently (limited by your semaphore)
        if scrape_tasks:
            await asyncio.gather(*scrape_tasks)

        # Mark the newest post as the top of the cache
        if top_post_id and top_post_id in cache:
            cache[top_post_id]['last_cached'] = True

        # Send final payload to UI if this was a user request
        if is_ui_request:
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            await safe_send(websocket, {"type": "progress", "value": 100})
            await safe_send(websocket, {"type": "final_data", "posts": final_selection})
            
    except Exception as e:
        print(f"Critical Scraper Error: {e}")
        if is_ui_request:
            await safe_send(websocket, {"type": "error", "message": str(e)})
    finally:
        await browser.close()
        await playwright.stop()

In [139]:
#await reanalyze_cache("LegalAdviceIndia")

In [140]:
async def get_cache_summary():
    """
    Fetches subreddit statistics and formats them as an array of objects:
    [{ subredditname: { count: X, last_updated: Y } }]
    """
    conn = await get_db_connection()
    try:
        # Optimized SQL aggregation
        rows = await conn.fetch('''
            SELECT 
                subreddit, 
                COUNT(*) as post_count, 
                MAX(timestamp) as last_post_time 
            FROM reddit_posts 
            GROUP BY subreddit
        ''')
        
        # Constructing the specific object format
        summary_dict = {}
        for row in rows:
            summary_dict[row['subreddit']] = {
                "count": row['post_count'],
                "last_updated": row['last_post_time'].isoformat() if row['last_post_time'] else None
            }
        return summary_dict
    finally:
        await conn.close()

In [141]:
async def handler(websocket):
    # --- NEW: Immediate Summary on Connection ---
    print("New connection established. Sending cache summary...")
    try:
        summary_data = await get_cache_summary()
        await websocket.send(json.dumps({
            "type": "cache_summary",
            "message": summary_data
        }))
    except Exception as e:
        print(f"Error sending summary: {e}")

    print("Waiting for command...")
    async for message in websocket:
        data = json.loads(message)
        if data.get('type') == 'start_scrape':
            sub = data.get('subreddit', 'mumbai')
            limit = data.get('count', 10)
            use_only_cache = data.get('useOnlyCache', False)
            
            print(f"Command received: r/{sub} (Limit: {limit}, Cache Only: {use_only_cache})")
            word_counts.clear()
            await run_scraper(websocket, sub, limit, use_only_cache)

# async def start_server():
#     print("WebSocket Server started on ws://192.168.0.246:8765")
#     async with websockets.serve(handler, "192.168.0.246", 8765):
#         await asyncio.Future()

#await start_server()


In [ ]:
async def background_worker():
    """Silently updates the database every 15 minutes."""
    print("Background worker initialized...")
    await asyncio.sleep(5) # Give the server a moment to start
    
    while True:
        for sub in background_subreddits:
            # If the user is actively clicking around in the UI, back off.
            if is_scraping.locked():
                print(f"UI in use. Background worker delaying r/{sub} update...")
                await asyncio.sleep(60)
                continue
                
            async with is_scraping:
                print(f"\n--- [BACKGROUND] Updating r/{sub} ---")
                try:
                    # Fetch max 15 posts silently in the background
                    await run_scraper(websocket=None, subreddit=sub, count=15, use_only_cache=False)
                except Exception as e:
                    print(f"Background error on r/{sub}: {e}")
            
            # Rest the CPU between subreddits
            await asyncio.sleep(10) 

        print(f"\n[BACKGROUND] Cycle complete. Sleeping for {SCRAPE_INTERVAL} seconds...")
        await asyncio.sleep(SCRAPE_INTERVAL)

async def handler(websocket):
    print("UI Client connected.")
    try:
        # Send initial cache summary to UI on load
        summary_data = await get_cache_summary()
        await safe_send(websocket, {"type": "cache_summary", "message": summary_data})
        
        async for message in websocket:
            data = json.loads(message)
            if data.get('type') == 'start_scrape':
                sub = data.get('subreddit', 'mumbai')
                limit = data.get('count', 10)
                use_only_cache = data.get('useOnlyCache', False)
                
                print(f"UI COMMAND: r/{sub} (Limit: {limit}, Cache Only: {use_only_cache})")
                
                # UI takes priority. Wait for lock, then execute.
                async with is_scraping:
                    await run_scraper(websocket, sub, limit, use_only_cache)
                    
    except websockets.exceptions.ConnectionClosed as e:
        print(f"UI Client disconnected: {e}")
    except Exception as e:
        print(f"WebSocket Handler Error: {e}")

async def start_services():
    print("Starting BI Data Pipeline...")
    
    # Start the background data collector
    bg_task = asyncio.create_task(background_worker())
    
    # Start the UI listener
    print("WebSocket Server active on ws://192.168.0.246:8765")
    async with websockets.serve(handler, "192.168.0.246", 8765):
        await asyncio.Future() # Keep running forever

# Start everything
await start_services()

Starting BI Data Pipeline...
WebSocket Server active on ws://192.168.0.246:8765
Background worker initialized...

--- [BACKGROUND] Updating r/Mumbai ---
Background error on r/Mumbai: BrowserType.launch: Executable doesn't exist at C:\Users\Administrator\AppData\Local\ms-playwright\chromium-1217\chrome-win64\chrome.exe
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     playwright install                                     ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝
UI Client connected.
UI Client connected.

--- [BACKGROUND] Updating r/India ---
Background error on r/India: BrowserType.launch: Executable doesn't exist at C:\Users\Administr